In [ ]:
# temperature heatmap
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import numpy as np

import os
import seaborn as sns
import pandas as pd

from statsmodels.tsa.seasonal import seasonal_decompose

DATA_DIR = os.path.join(os.getenv('DATASET_DIR'), 'WeatherBench/5.625deg')
SST_DATA_PATH = os.path.join(DATA_DIR, 'temperature_850', '*.nc')

data = xr.open_mfdataset(SST_DATA_PATH, combine='by_coords').to_array()
# resampled_data = data.resample(time='2D').mean().squeeze()
# data = data.sel(lon=slice(45, 75), lat=slice(45, 75)).stack(space=('lat', 'lon')).resample(time=resample_size).mean().squeeze()
# fig = plt.figure(figsize=(10, 5))
# ax = plt.axes(projection=ccrs.PlateCarree())
# ax.coastlines()
# ax.gridlines(draw_labels=True)
# data.plot(ax=ax, transform=ccrs.PlateCarree(), x='lon', y='lat')

In [ ]:
#sel_data = data.sel(lon=slice(15, 85), lat=slice(15, 60)).stack(space=('lat', 'lon')).values
#sel_data = data.sel(lon=slice(65, 85), lat=slice(25, 45)).stack(space=('lat', 'lon')).values
#sel_data = data.sel(lon=slice(45, 85), lat=slice(15, 45)).stack(space=('lat', 'lon')).values

In [ ]:
sel_data = data.sel(lon=slice(45, 85), lat=slice(15, 45)).stack(space=('lat', 'lon')).values.shape
sel_data

In [ ]:
np.save('./350640_16.npy', sel_data[0])   

In [ ]:
sel_data[0].shape

In [ ]:
fig = plt.figure(figsize=(10, 5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
ax.gridlines(draw_labels=True)
data.sel(lon=slice(15, 85), lat=slice(15, 60)).plot(ax=ax, transform=ccrs.PlateCarree(), x='lon', y='lat')

In [ ]:
xr.open_mfdataset(SST_DATA_PATH, combine='by_coords').to_array().sel(lon=slice(15, 85), lat=slice(15, 60)).shape

In [ ]:
u = xr.open_mfdataset(os.path.join(DATA_DIR, '10m_u_component_of_wind', '*.nc'), combine='by_coords')
v = xr.open_mfdataset(os.path.join(DATA_DIR, '10m_v_component_of_wind', '*.nc'), combine='by_coords')
u.to_array().values

In [ ]:
resample_size='1D'
data.sel(lon=slice(45, 75), lat=slice(45, 75)).stack(space=('lat', 'lon')).resample(time=resample_size).mean().squeeze() 

In [ ]:
# Assuming 'data' is your xarray DataArray
temperature_values_in_kelvin = data.values

# Convert to degrees Celsius
temperature_values_in_celsius = temperature_values_in_kelvin - 273.15

print(temperature_values_in_celsius)

In [ ]:
import numpy as np

months = np.arange('1973-05', '2000-11', dtype='datetime64[M]')
temperature = temperature_values_in_celsius[:len(months)]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(months, temperature, marker='o', linestyle='-', color='b')

ax.set_title('Temperature Trend from May 1973 to October 1974', fontsize=14, fontweight='bold')
ax.set_xlabel('Month', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)

ax.tick_params(axis='both', which='major', labelsize=10)
ax.grid(True, linestyle='--', linewidth=0.5)

ax.legend(loc='upper right')

# Format the x-axis for better readability
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=24))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

fig.autofmt_xdate()

# Save the figure
plt.savefig('temperature_trend.png', dpi=300, bbox_inches='tight')

# Show the plot
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import periodogram
import matplotlib.pyplot as plt

# Assuming 'data' is a list of values
data_frame = pd.DataFrame(temperature, columns=['Value'])

# Calculate the periodogram
frequencies, power = periodogram(data_frame['Value'])

# Find the frequency with the highest power
dominant_frequency = frequencies[np.argmax(power)]

# The dominant period is the inverse of the dominant frequency
dominant_period = 1 / dominant_frequency

print(f"The dominant period is: {dominant_period}")

# Plot the periodogram
plt.figure(figsize=(10, 6))
plt.plot(frequencies, power, color='dodgerblue')
plt.title('Periodogram')
plt.xlabel('Frequency')
plt.ylabel('Power')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import periodogram, argrelextrema
import matplotlib.pyplot as plt

# Assuming 'data' is a list of values
data_frame = pd.DataFrame(temperature, columns=['Value'])

# Calculate the periodogram
frequencies, power = periodogram(data_frame['Value'])

# Find the local maxima in the power spectrum
local_maxima = argrelextrema(power, np.greater)

# The periods corresponding to the local maxima are the inverses of the frequencies
periods = 1 / frequencies[local_maxima]

print(f"The periods are: {periods}")

# Plot the periodogram
plt.figure(figsize=(10, 6))
plt.plot(frequencies, power)
plt.scatter(frequencies[local_maxima], power[local_maxima], color='red')  # highlight the local maxima
plt.title('Periodogram')
plt.xlabel('Frequency')
plt.ylabel('Power')
plt.show()

In [ ]:
# temperature heatmap
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

z500 = xr.open_mfdataset('${DATA_DIR}/WeatherBench_data_full/temperature_850/*.nc', combine='by_coords')
data = z500.t.isel(time=300000)

# Plotting
fig = plt.figure(figsize=(10, 5))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.coastlines()
#ax.set_extent([-100, 20, 0, 70])

# Plot the data
ax.gridlines(draw_labels=True)
data.plot(ax=ax, transform=ccrs.PlateCarree(), x='lon', y='lat')